# Seedwarden Cohort Analysis Dashboard

**Purpose**: Monthly cohort visualization for growth analysis.  
**Data sources**: Etsy Stats CSV exports + Kit (ConvertKit) subscriber export.  
**Run cadence**: First Monday of each month, using prior month's export files.  

**Sections**:
1. Data loading and cleaning
2. Revenue and order volume trends
3. Cohort retention heatmap
4. LTV curves by cohort
5. Seasonal cohort performance
6. Product-level conversion rates
7. Email engagement health
8. Monthly executive summary

## 0. Setup

In [ ]:
# Install requirements (run once):
# uv pip install pandas matplotlib seaborn duckdb openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import duckdb
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ---- Configuration ----
DATA_DIR = Path('../data')          # Place Etsy/Kit CSV exports here
REPORTING_MONTH = '2026-05'         # Update each month: 'YYYY-MM'
ETSY_FEES_PCT = 0.095               # 6.5% transaction + 3% payment processing
ETSY_PAYMENT_FLAT = 0.25            # $0.25 flat per transaction

# Brand palette
COLORS = {
    'primary':   '#3d6b45',   # Forest green
    'secondary': '#8b6914',   # Harvest gold
    'accent':    '#c4503a',   # Terracotta
    'neutral':   '#f5f0e8',   # Parchment
    'text':      '#2c2c2c',
}
SEASONAL_COLORS = {
    'spring_planning': '#3d6b45',
    'preservation':    '#8b6914',
    'holiday_gift':    '#c4503a',
    'long_tail_spring':'#7ab38c',
    'long_tail_fall':  '#c8a96e',
}

print(f'Seedwarden Dashboard — Reporting Month: {REPORTING_MONTH}')
print(f'Data directory: {DATA_DIR.resolve()}')

## 1. Data Loading and Cleaning

In [ ]:
# ---- 1.1 Load Etsy orders export ----
# Etsy provides: Shop Manager > Finances > Payment Account > Download CSV
# Columns typically: Order ID, Order Date, Order Value, Quantity, Item, Buyer Email, etc.

def load_etsy_orders(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=['Order Date'])
    
    # Normalize column names
    df.columns = (
        df.columns.str.strip()
                  .str.lower()
                  .str.replace(' ', '_', regex=False)
                  .str.replace('[^a-z0-9_]', '', regex=True)
    )
    
    # Derive net revenue
    df['gross_revenue'] = pd.to_numeric(df.get('order_total', df.get('item_total')), errors='coerce')
    df['etsy_fees']     = (df['gross_revenue'] * ETSY_FEES_PCT + ETSY_PAYMENT_FLAT).round(2)
    df['net_revenue']   = (df['gross_revenue'] - df['etsy_fees']).round(2)
    
    # Parse order date robustly
    date_col = next((c for c in df.columns if 'date' in c), None)
    if date_col:
        df['order_date'] = pd.to_datetime(df[date_col], errors='coerce')
    
    df['order_ym'] = df['order_date'].dt.to_period('M').astype(str)
    
    # Acquisition season
    month = df['order_date'].dt.month
    conditions = [
        month.isin([1, 2, 3, 4]),
        month.isin([5, 6]),
        month.isin([7, 8, 9]),
        month.isin([10]),
        month.isin([11, 12]),
    ]
    choices = [
        'spring_planning', 'long_tail_spring',
        'preservation',    'long_tail_fall', 'holiday_gift'
    ]
    df['acquisition_season'] = np.select(conditions, choices, default='unknown')
    
    return df


# Load if file exists; otherwise create sample data for template validation
orders_path = DATA_DIR / 'etsy_orders.csv'
if orders_path.exists():
    orders = load_etsy_orders(orders_path)
    print(f'Loaded {len(orders):,} orders from {orders_path}')
else:
    print('No etsy_orders.csv found — generating synthetic sample data for layout validation.')
    # ---- Synthetic sample (replace with real data) ----
    rng = np.random.default_rng(42)
    n = 250
    start = pd.Timestamp('2026-05-01')
    dates = [start + timedelta(days=int(rng.uniform(0, 365))) for _ in range(n)]
    emails = [f'buyer_{rng.integers(1, 120)}@example.com' for _ in range(n)]
    products = rng.choice(
        ['Companion Planting Chart', 'Food Sovereignty Guide', 'Seed Saving Manual',
         'Preservation Bundle', 'Apartment Grower Bundle', 'Native Plants Guide',
         'Harvest Preservation', 'Zone Calendar', 'Survival Garden Plans'],
        size=n
    )
    prices = {
        'Companion Planting Chart': 5, 'Food Sovereignty Guide': 8,
        'Zone Calendar': 8, 'Seed Saving Manual': 14,
        'Harvest Preservation': 16, 'Native Plants Guide': 18,
        'Survival Garden Plans': 22, 'Apartment Grower Bundle': 32,
        'Preservation Bundle': 38,
    }
    channels = rng.choice(
        ['etsy_organic', 'etsy_ads', 'email_utm', 'pinterest_utm', 'influencer_code'],
        size=n, p=[0.50, 0.20, 0.15, 0.10, 0.05]
    )
    gross = np.array([prices[p] for p in products], dtype=float)
    orders = pd.DataFrame({
        'order_id':          [f'ORD-{i:04d}' for i in range(n)],
        'buyer_email':        emails,
        'order_date':         pd.to_datetime(dates),
        'product_name':       products,
        'gross_revenue':      gross,
        'etsy_fees':          (gross * ETSY_FEES_PCT + ETSY_PAYMENT_FLAT).round(2),
        'net_revenue':        (gross - gross * ETSY_FEES_PCT - ETSY_PAYMENT_FLAT).round(2),
        'channel':            channels,
        'is_bundle':          [p.endswith('Bundle') for p in products],
    })
    orders['order_ym'] = orders['order_date'].dt.to_period('M').astype(str)
    month = orders['order_date'].dt.month
    orders['acquisition_season'] = np.select(
        [month.isin([1,2,3,4]), month.isin([5,6]),
         month.isin([7,8,9]),  month.isin([10]),   month.isin([11,12])],
        ['spring_planning','long_tail_spring','preservation','long_tail_fall','holiday_gift'],
        default='unknown'
    )
    print(f'Generated {n} synthetic orders.')

print(orders[['order_date','buyer_email','gross_revenue','channel','acquisition_season']].head())

In [ ]:
# ---- 1.2 Build first-purchase cohort table ----

first_purchase = (
    orders.sort_values('order_date')
          .groupby('buyer_email')
          .first()
          .reset_index()
          [['buyer_email', 'order_date', 'order_ym', 'product_name',
            'gross_revenue', 'acquisition_season', 'channel']]
          .rename(columns={
              'order_date':    'first_order_date',
              'order_ym':      'cohort_ym',
              'product_name':  'first_product',
              'gross_revenue': 'first_order_value',
              'channel':       'acquisition_channel',
          })
)

# Price tier of first product
def price_tier(price):
    if price <= 9.99:  return 'entry ($5-$9)'
    if price <= 15.99: return 'mid ($10-$15)'
    if price <= 29.99: return 'premium ($16-$22)'
    return 'bundle ($28-$50)'

first_purchase['price_tier'] = first_purchase['first_order_value'].apply(price_tier)

# Merge cohort info back onto all orders
orders = orders.merge(
    first_purchase[['buyer_email', 'first_order_date', 'cohort_ym', 'price_tier']],
    on='buyer_email', how='left'
)
orders['months_since_first'] = (
    (orders['order_date'].dt.to_period('M') -
     orders['first_order_date'].dt.to_period('M')).apply(lambda x: x.n if x else 0)
)

print(f'Unique buyers: {orders["buyer_email"].nunique()}')
print(f'Cohort months: {sorted(orders["cohort_ym"].unique())}')

## 2. Revenue and Order Volume Trends

In [ ]:
monthly = (
    orders.groupby('order_ym')
          .agg(
              gross_revenue=('gross_revenue', 'sum'),
              net_revenue=('net_revenue', 'sum'),
              orders=('order_id', 'count'),
              unique_buyers=('buyer_email', 'nunique'),
          )
          .reset_index()
          .sort_values('order_ym')
)
monthly['aov'] = (monthly['gross_revenue'] / monthly['orders']).round(2)
# Bundle revenue share
bundle_monthly = (
    orders[orders.get('is_bundle', False)]
         .groupby('order_ym')['gross_revenue'].sum()
         .rename('bundle_revenue')
)
monthly = monthly.join(bundle_monthly, on='order_ym').fillna(0)
monthly['bundle_pct'] = (monthly['bundle_revenue'] / monthly['gross_revenue'] * 100).round(1)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Seedwarden — Monthly Revenue Dashboard', fontsize=14, color=COLORS['text'])

# Gross vs net revenue
ax = axes[0, 0]
x = range(len(monthly))
ax.bar(x, monthly['gross_revenue'], label='Gross', color=COLORS['primary'], alpha=0.8)
ax.bar(x, monthly['net_revenue'],   label='Net',   color=COLORS['secondary'], alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(monthly['order_ym'], rotation=45, ha='right', fontsize=8)
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))
ax.set_title('Gross vs Net Revenue'); ax.legend(fontsize=8)

# Orders and unique buyers
ax = axes[0, 1]
ax.bar(x, monthly['orders'],        label='Orders',  color=COLORS['primary'], alpha=0.7)
ax.bar(x, monthly['unique_buyers'], label='Buyers',  color=COLORS['accent'],  alpha=0.7)
ax.set_xticks(x); ax.set_xticklabels(monthly['order_ym'], rotation=45, ha='right', fontsize=8)
ax.set_title('Orders and Unique Buyers'); ax.legend(fontsize=8)

# Average order value
ax = axes[1, 0]
ax.plot(x, monthly['aov'], color=COLORS['secondary'], marker='o', linewidth=2)
ax.axhline(monthly['aov'].mean(), color=COLORS['accent'], linestyle='--', label=f'Mean ${monthly["aov"].mean():.2f}')
ax.set_xticks(x); ax.set_xticklabels(monthly['order_ym'], rotation=45, ha='right', fontsize=8)
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter('${x:.0f}')); ax.set_title('Average Order Value')
ax.legend(fontsize=8)

# Bundle revenue % of total
ax = axes[1, 1]
ax.bar(x, monthly['bundle_pct'], color=COLORS['accent'], alpha=0.8)
ax.axhline(35, color=COLORS['secondary'], linestyle='--', label='Target: 35%')
ax.set_xticks(x); ax.set_xticklabels(monthly['order_ym'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('%'); ax.set_title('Bundle Revenue as % of Total'); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(DATA_DIR / 'output_revenue_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print(monthly.tail(6).to_string(index=False))

## 3. Cohort Retention Heatmap

In [ ]:
# Build retention matrix: rows = cohort month, cols = months since first purchase

cohort_retention = (
    orders.groupby(['cohort_ym', 'months_since_first'])['buyer_email']
          .nunique()
          .reset_index()
          .rename(columns={'buyer_email': 'active_buyers'})
)

cohort_sizes = (
    cohort_retention[cohort_retention['months_since_first'] == 0]
    [['cohort_ym', 'active_buyers']]
    .rename(columns={'active_buyers': 'cohort_size'})
)

cohort_retention = cohort_retention.merge(cohort_sizes, on='cohort_ym')
cohort_retention['retention_pct'] = (
    cohort_retention['active_buyers'] / cohort_retention['cohort_size'] * 100
).round(1)

retention_pivot = cohort_retention.pivot(
    index='cohort_ym', columns='months_since_first', values='retention_pct'
)
# Add cohort size label
size_labels = cohort_sizes.set_index('cohort_ym')['cohort_size']
retention_pivot.index = [
    f"{ym} (n={size_labels.get(ym, 0)})" for ym in retention_pivot.index
]

fig, ax = plt.subplots(figsize=(max(10, len(retention_pivot.columns) + 2),
                                max(5,  len(retention_pivot) + 1)))

# Custom colormap: green gradient
cmap = sns.light_palette(COLORS['primary'], as_cmap=True)

sns.heatmap(
    retention_pivot,
    annot=True, fmt='.0f', cmap=cmap,
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Retention %', 'format': '%.0f%%'},
    ax=ax
)
ax.set_title('Customer Cohort Retention Heatmap\n(% of cohort making a purchase in each subsequent month)',
             fontsize=12, color=COLORS['text'], pad=12)
ax.set_xlabel('Months Since First Purchase', fontsize=10)
ax.set_ylabel('Acquisition Cohort (YYYY-MM)', fontsize=10)

# Annotate Month 0 column (always 100%) as baseline
ax.annotate('Month 0\n= acquisition', xy=(0.5, -0.12), xycoords='axes fraction',
            ha='center', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig(DATA_DIR / 'output_cohort_retention_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. LTV Curves by First-Product Price Tier

In [ ]:
# Cumulative LTV curve: average cumulative net revenue per buyer by months since acquisition
# Faceted by first-product price tier

ltv_by_tier = (
    orders.groupby(['price_tier', 'months_since_first'])['net_revenue']
          .sum()
          .reset_index()
)

# Divide by cohort size per tier to get per-buyer average
tier_sizes = first_purchase.groupby('price_tier').size().rename('tier_size').reset_index()
ltv_by_tier = ltv_by_tier.merge(tier_sizes, on='price_tier')
ltv_by_tier['revenue_per_buyer'] = ltv_by_tier['net_revenue'] / ltv_by_tier['tier_size']

# Cumulative sum within each tier
ltv_by_tier = ltv_by_tier.sort_values(['price_tier', 'months_since_first'])
ltv_by_tier['cumulative_ltv'] = (
    ltv_by_tier.groupby('price_tier')['revenue_per_buyer'].cumsum()
)

fig, ax = plt.subplots(figsize=(12, 6))

tier_colors = {
    'entry ($5-$9)':    COLORS['neutral'],
    'mid ($10-$15)':    COLORS['secondary'],
    'premium ($16-$22)':COLORS['primary'],
    'bundle ($28-$50)': COLORS['accent'],
}

for tier, group in ltv_by_tier.groupby('price_tier'):
    ax.plot(
        group['months_since_first'], group['cumulative_ltv'],
        label=tier, linewidth=2.5,
        color=tier_colors.get(tier, 'gray'),
        marker='o', markersize=4
    )

# LTV targets from framework
targets = {'entry ($5-$9)': 14.80, 'mid ($10-$15)': 21.80,
           'premium ($16-$22)': 31.50, 'bundle ($28-$50)': 43.00}
for tier, target in targets.items():
    ax.axhline(target, color=tier_colors.get(tier, 'gray'),
               linestyle='--', alpha=0.4, linewidth=1)

ax.yaxis.set_major_formatter(mticker.StrMethodFormatter('${x:.0f}'))
ax.set_xlabel('Months Since First Purchase', fontsize=11)
ax.set_ylabel('Cumulative LTV per Buyer (Net Revenue)', fontsize=11)
ax.set_title('Customer LTV Curves by First-Product Price Tier\n(dashed lines = 24-month targets)',
             fontsize=12, color=COLORS['text'])
ax.legend(title='First Purchase Tier', fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_DIR / 'output_ltv_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Seasonal Cohort Performance

In [ ]:
# 5.1  Revenue by season and calendar month
seasonal_monthly = (
    orders.groupby(['acquisition_season', 'order_ym'])['net_revenue']
          .sum()
          .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Revenue by month, stacked by season
pivot_rev = seasonal_monthly.pivot_table(
    index='order_ym', columns='acquisition_season',
    values='net_revenue', fill_value=0
)
pivot_rev.plot(
    kind='bar', stacked=True, ax=axes[0],
    color=[SEASONAL_COLORS.get(c, '#888') for c in pivot_rev.columns]
)
axes[0].set_title('Monthly Net Revenue by Acquisition Season', fontsize=11)
axes[0].set_xlabel('')
axes[0].yaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(title='Acquisition Season', fontsize=8, loc='upper left')

# 5.2  Second-purchase rate by seasonal cohort
orders_ranked = orders.sort_values(['buyer_email', 'order_date'])
orders_ranked['purchase_rank'] = orders_ranked.groupby('buyer_email').cumcount() + 1

second_purch = (
    orders_ranked[orders_ranked['purchase_rank'] == 2]
    .groupby('buyer_email')['order_date'].min()
    .reset_index()
    .rename(columns={'order_date': 'second_order_date'})
)

seasonal_analysis = first_purchase.merge(
    second_purch, on='buyer_email', how='left'
)
seasonal_analysis['had_second_purchase'] = seasonal_analysis['second_order_date'].notna()
seasonal_analysis['days_to_second'] = (
    seasonal_analysis['second_order_date'] - seasonal_analysis['first_order_date']
).dt.days
seasonal_analysis['second_purchase_within_90d'] = seasonal_analysis['days_to_second'] <= 90

season_rates = (
    seasonal_analysis
    .groupby('acquisition_season')
    .agg(
        cohort_size=('buyer_email', 'count'),
        any_second_purchase=('had_second_purchase', 'sum'),
        second_within_90d=('second_purchase_within_90d', 'sum'),
    )
    .assign(
        repeat_rate_pct=lambda d: (d['any_second_purchase'] / d['cohort_size'] * 100).round(1),
        repeat_90d_rate_pct=lambda d: (d['second_within_90d'] / d['cohort_size'] * 100).round(1),
    )
    .reset_index()
)

x = range(len(season_rates))
width = 0.35
axes[1].bar([i - width/2 for i in x], season_rates['repeat_rate_pct'],
            width, label='Any second purchase',
            color=[SEASONAL_COLORS.get(s, '#888') for s in season_rates['acquisition_season']],
            alpha=0.8)
axes[1].bar([i + width/2 for i in x], season_rates['repeat_90d_rate_pct'],
            width, label='Within 90 days',
            color=[SEASONAL_COLORS.get(s, '#888') for s in season_rates['acquisition_season']],
            alpha=0.4)

# Target line
axes[1].axhline(20, color=COLORS['accent'], linestyle='--', label='Target 20%', linewidth=1.5)

axes[1].set_xticks(x)
axes[1].set_xticklabels(season_rates['acquisition_season'], rotation=20, ha='right', fontsize=9)
axes[1].set_ylabel('%'); axes[1].set_title('Second-Purchase Rate by Seasonal Cohort', fontsize=11)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(DATA_DIR / 'output_seasonal_cohorts.png', dpi=150, bbox_inches='tight')
plt.show()
print(season_rates[['acquisition_season','cohort_size','repeat_rate_pct','repeat_90d_rate_pct']].to_string(index=False))

## 6. Product-Level Conversion Rates

In [ ]:
# Product revenue and repeat-trigger rate

product_sales = (
    orders.groupby('product_name')
          .agg(
              total_revenue=('gross_revenue', 'sum'),
              total_orders=('order_id', 'count'),
              unique_buyers=('buyer_email', 'nunique'),
          )
          .reset_index()
          .sort_values('total_revenue', ascending=False)
)

# Repeat-trigger rate: of buyers whose FIRST purchase was this product,
# what proportion made any subsequent purchase?
first_trigger = (
    first_purchase.merge(
        second_purch[['buyer_email', 'second_order_date']], on='buyer_email', how='left'
    )
    .groupby('first_product')
    .agg(
        first_buyers=('buyer_email', 'count'),
        repeat_buyers=('second_order_date', lambda x: x.notna().sum()),
    )
    .assign(repeat_trigger_pct=lambda d: (d['repeat_buyers'] / d['first_buyers'] * 100).round(1))
    .reset_index()
    .rename(columns={'first_product': 'product_name'})
)

product_summary = product_sales.merge(first_trigger, on='product_name', how='left')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Revenue by product
ax = axes[0]
ps = product_summary.sort_values('total_revenue', ascending=True)
bars = ax.barh(ps['product_name'], ps['total_revenue'],
               color=COLORS['primary'], alpha=0.8)
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))
ax.set_title('Total Revenue by Product', fontsize=11)

# Repeat-trigger rate by product
ax = axes[1]
tr = product_summary.dropna(subset=['repeat_trigger_pct'])\
                    .sort_values('repeat_trigger_pct', ascending=True)
colors_bar = [COLORS['primary'] if v >= 20 else COLORS['secondary'] if v >= 10
              else COLORS['accent'] for v in tr['repeat_trigger_pct']]
ax.barh(tr['product_name'], tr['repeat_trigger_pct'], color=colors_bar, alpha=0.8)
ax.axvline(20, color=COLORS['text'], linestyle='--', linewidth=1.5,
           label='Target 20% (repeat-purchase driver)')
ax.set_xlabel('Repeat-Purchase Trigger Rate (%)')
ax.set_title('Repeat-Purchase Trigger Rate by First Product\n(buyers who made a second purchase)', fontsize=11)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(DATA_DIR / 'output_product_performance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 products by revenue:')
print(product_summary[['product_name','total_revenue','total_orders',
                         'repeat_trigger_pct']].head(10).to_string(index=False))

## 7. Email Engagement Health

This section loads Kit subscriber data. If the Kit export is not available,
the cell generates a synthetic dataset for layout validation.

In [ ]:
# ---- Load Kit subscriber export ----
# Kit export: Subscribers > Export > CSV
# Required columns: email, subscribed_at, tags, total_email_count, open_count

kit_path = DATA_DIR / 'kit_subscribers.csv'

if kit_path.exists():
    subs = pd.read_csv(kit_path, parse_dates=['subscribed_at'])
    subs.columns = subs.columns.str.strip().str.lower().str.replace(' ', '_')
    print(f'Loaded {len(subs):,} subscribers from Kit export.')
else:
    print('No kit_subscribers.csv found — generating synthetic subscriber data.')
    rng = np.random.default_rng(99)
    n_subs = 480
    subs = pd.DataFrame({
        'email':          [f'sub_{i}@example.com' for i in range(n_subs)],
        'subscribed_at':  pd.to_datetime('2026-05-01') + pd.to_timedelta(
                              rng.integers(0, 360, n_subs), unit='D'),
        'tags':           rng.choice(
                              ['seed-saver', 'city-grower', 'preservationist', 'new-subscriber', ''],
                              size=n_subs, p=[0.25, 0.30, 0.20, 0.15, 0.10]),
        'total_email_count': rng.integers(3, 30, n_subs),
        'open_count':        rng.integers(0, 20, n_subs),
    })

# Health tier classification
subs['open_rate'] = (subs['open_count'] / subs['total_email_count'].replace(0, np.nan)).fillna(0)
subs['health_tier'] = pd.cut(
    subs['open_rate'],
    bins=[-0.01, 0.15, 0.30, 1.01],
    labels=['cold (<15%)', 'marginal (15-30%)', 'engaged (30%+)']
)

tier_counts = subs['health_tier'].value_counts()
tier_pcts   = (tier_counts / len(subs) * 100).round(1)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Health tier donut
ax = axes[0]
donut_colors = [COLORS['accent'], COLORS['secondary'], COLORS['primary']]
wedges, texts, autotexts = ax.pie(
    tier_counts, labels=tier_counts.index, autopct='%1.0f%%',
    colors=donut_colors, wedgeprops={'width': 0.5}, startangle=90
)
for t in autotexts: t.set_fontsize(10)
ax.set_title(f'Subscriber Health Distribution\n(n={len(subs):,} total)', fontsize=11)

# List growth over time
ax = axes[1]
growth = (
    subs.groupby(subs['subscribed_at'].dt.to_period('M').astype(str))
        .size()
        .reset_index()
        .rename(columns={0: 'new_subs', 'subscribed_at': 'month'})
)
growth['cumulative'] = growth['new_subs'].cumsum()
ax2 = ax.twinx()
ax.bar(range(len(growth)), growth['new_subs'], color=COLORS['primary'], alpha=0.6, label='New subs')
ax2.plot(range(len(growth)), growth['cumulative'], color=COLORS['accent'],
         linewidth=2, marker='o', label='Cumulative')
ax.set_xticks(range(len(growth)))
ax.set_xticklabels(growth['month'], rotation=45, ha='right', fontsize=8)
ax.set_title('Email List Growth', fontsize=11)
ax.set_ylabel('New Subscribers / Month'); ax2.set_ylabel('Total Subscribers')

# Segment tag distribution
ax = axes[2]
tag_counts = subs['tags'].value_counts().head(6)
ax.bar(range(len(tag_counts)), tag_counts.values,
       color=[COLORS['primary'], COLORS['secondary'], COLORS['accent'],
              COLORS['primary'], COLORS['secondary'], COLORS['accent']], alpha=0.8)
ax.set_xticks(range(len(tag_counts)))
ax.set_xticklabels(tag_counts.index, rotation=30, ha='right', fontsize=9)
ax.set_title('Behavioral Tag Distribution', fontsize=11)

plt.tight_layout()
plt.savefig(DATA_DIR / 'output_email_health.png', dpi=150, bbox_inches='tight')
plt.show()

print('Subscriber health summary:')
for tier, count in tier_counts.items():
    pct = tier_pcts[tier]
    status = 'OK' if tier == 'engaged (30%+)' else ('WATCH' if tier == 'marginal (15-30%)' else 'ACT')
    print(f'  [{status}] {tier}: {count:,} ({pct}%)')

## 8. Monthly Executive Summary

In [ ]:
# Scorecard against 6-month and 12-month targets from growth-metrics-framework.md

MONTHS_SINCE_LAUNCH = 1  # Update each month

current_month_orders = orders[orders['order_ym'] == REPORTING_MONTH]

# Metrics
mtd_gross        = current_month_orders['gross_revenue'].sum()
mtd_net          = current_month_orders['net_revenue'].sum()
mtd_orders_count = len(current_month_orders)
mtd_aov          = mtd_gross / mtd_orders_count if mtd_orders_count else 0
bundle_rev_pct   = (current_month_orders.get('is_bundle', pd.Series(False)).sum() /
                    mtd_orders_count * 100) if mtd_orders_count else 0

total_subs       = len(subs)
engaged_pct      = (subs['health_tier'] == 'engaged (30%+)').mean() * 100
second_purch_rate = (seasonal_analysis['had_second_purchase'].mean() * 100
                     if 'seasonal_analysis' in dir() else 0)

# Targets from framework
targets_6m = {
    'Monthly Gross Revenue': (600, 1200),
    'Email Subscribers':     (400, 600),
    'Email Open Rate (%)':   (32, 100),
    'Second-Purchase Rate (%)': (15, 100),
}

actuals = {
    'Monthly Gross Revenue': mtd_gross,
    'Email Subscribers':     total_subs,
    'Email Open Rate (%)':   engaged_pct,
    'Second-Purchase Rate (%)': second_purch_rate,
}

print('=' * 60)
print(f'SEEDWARDEN MONTHLY SCORECARD — {REPORTING_MONTH}')
print(f'Months since launch: {MONTHS_SINCE_LAUNCH}')
print('=' * 60)
print(f'  Gross Revenue (MTD):     ${mtd_gross:>8,.2f}')
print(f'  Net Revenue (MTD):       ${mtd_net:>8,.2f}')
print(f'  Orders (MTD):            {mtd_orders_count:>8}')
print(f'  Average Order Value:     ${mtd_aov:>8,.2f}')
print(f'  Bundle Revenue %:        {bundle_rev_pct:>7.1f}%  (target: 35%)')
print('-' * 60)
print(f'  Email Subscribers:       {total_subs:>8,}')
print(f'  Engaged Subscriber %:    {engaged_pct:>7.1f}%  (target: 30%+)')
print(f'  Second-Purchase Rate:    {second_purch_rate:>7.1f}%  (target: 20%+)')
print('=' * 60)
print()
print('Target tracking (6-month targets):')
for metric, (lo, hi) in targets_6m.items():
    actual = actuals[metric]
    if actual >= lo:
        status = 'ON TRACK'
    elif actual >= lo * 0.7:
        status = 'BEHIND'
    else:
        status = 'ACTION NEEDED'
    print(f'  [{status:<14}] {metric}: {actual:.1f} (target: {lo}-{hi})')

print('\nAll charts saved to data/output_*.png')